# SDMX objects → VTL with `run_sdmx`

The walkthrough fed VTL as a *script string* with JSON data structures. When you already work with **SDMX**, `run_sdmx` lets you stay in the pysdmx object model instead: hand it a [`TransformationScheme`](https://py.sdmx.io/) (the SDMX artefact that carries VTL transformations) together with one or more [`PandasDataset`](https://py.sdmx.io/) objects (a pandas DataFrame described by an SDMX `Schema`).

In this notebook we:

1. **Download** a real dataflow structure — ESTAT `ECB_QSA_Q` — from the SDMX Global Registry.
2. Turn its data structure into a **`PandasDataset`** with a few sample observations.
3. Define a **`TransformationScheme`** that derives a new measure.
4. Execute it with **`run_sdmx`**.

> **Networking:** step 1 fetches a remote URL. It works in this in-browser demo because the registry allows cross-origin requests and the demo auto-loads `ssl` + `certifi`.

## 1. Download a dataflow structure

`read_sdmx` accepts a URL. We request the `ECB_QSA_Q` dataflow with `references=children`, which also pulls in the data structure definition (DSD) it relies on. From the message we take that DSD.

In [ ]:
from pysdmx.io import read_sdmx

url = (
    "https://registry.sdmx.org/sdmx/v2/structure/dataflow/"
    "ESTAT/ECB_QSA_Q/+/?format=sdmx-3.0&references=children"
)
msg = read_sdmx(url)
dsd = msg.get_data_structure_definitions()[0]
print(f"{dsd.short_urn} — {len(dsd.components)} components")

## 2. Build a `PandasDataset`

A `PandasDataset` pairs a pandas DataFrame with an SDMX `Schema`, which we build straight from the DSD's components. Every **dimension** is an identifier and must be present and non-null, so we fill the breakdown dimensions with a placeholder code and vary `REF_AREA` / `TIME_PERIOD` to get three distinct observations.

In [ ]:
import pandas as pd
from pysdmx.io.pd import PandasDataset
from pysdmx.model import Role, Schema

schema = Schema(
    agency=dsd.agency,
    id=dsd.id,
    version=dsd.version,
    components=dsd.components,
    context="datastructure",
)

dimensions = [c.id for c in dsd.components if c.role == Role.DIMENSION]
base = {d: "_T" for d in dimensions if d != "TIME_PERIOD"}
rows = [
    {**base, "REF_AREA": "EA20", "TIME_PERIOD": "2022-Q1", "OBS_VALUE": 100.0},
    {**base, "REF_AREA": "EA20", "TIME_PERIOD": "2022-Q2", "OBS_VALUE": 110.0},
    {**base, "REF_AREA": "US", "TIME_PERIOD": "2022-Q1", "OBS_VALUE": 90.0},
]
dataset = PandasDataset(structure=schema, data=pd.DataFrame(rows))
dataset.data[["REF_AREA", "TIME_PERIOD", "OBS_VALUE"]]

## 3. Define a `TransformationScheme`

A `TransformationScheme` holds an ordered list of `Transformation`s. Each has a `result` (the output dataset name) and an `expression` (the VTL right-hand side). Here a single transformation adds a derived measure `Me_doubled`.

In [ ]:
from pysdmx.model import Transformation, TransformationScheme

ts = TransformationScheme(
    id="DEMO",
    version="1.0",
    agency="MD",
    vtl_version="2.1",
    items=[
        Transformation(
            id="T1",
            expression="ECB_QSA_Q [calc Me_doubled := OBS_VALUE * 2]",
            result="DS_r",
            is_persistent=False,
        )
    ],
)
print(f"{ts.items[0].result} <- {ts.items[0].expression}")

## 4. Execute with `run_sdmx`

`run_sdmx` takes the `TransformationScheme` and the list of datasets. With a **single** dataset and a **single** input it auto-matches them, so the input name in the expression (`ECB_QSA_Q`) is bound to our dataset automatically. For more than one dataset, pass an explicit `mappings` argument.

In [ ]:
from vtlengine import run_sdmx

result = run_sdmx(ts, [dataset], return_only_persistent=False)
result["DS_r"].data[["REF_AREA", "TIME_PERIOD", "OBS_VALUE", "Me_doubled"]]

The output dataset keeps every identifier from the input and adds the computed `Me_doubled` measure.

See also the [walkthrough](walkthrough.ipynb) for the script-string API, or [vtl-demo](vtl-demo.ipynb) for a minimal start.